# Round-1 revision checks (JCP, jcp-4679542)

**Run All.** One progress line is updated while it runs; **one report** is printed at the end (also saved with a JSON of all numbers to `release/audit/results/revision_r1/`). Expected run time ≈ 1 hour. Model predictions are cached, so a re-run after an interruption does not retrain finished models.

Uses the protocol of the revised paper: UNSW-NB15 chronological split (Stime 0.70/0.85); CIC-IDS2017 corrected (AM/PM) timestamps and stratified temporal split; LightGBM frozen configuration (num_leaves 15, lr 0.05, min_data_in_leaf 50), T = 500 rounds, no early stopping; 1-minute block bootstrap, B = 500.

| Section | What it checks | Reviewer comments |
|---|---|---|
| **0** | **Verification of the frozen results of the paper** (88 values: split sizes, scale_pos_weight, AP / ROC-AUC / recall of F0, F1 and both port models, the bootstrap CIs of Tables 10, 13 and 17 re-created with their original RNG streams, per-class ΔAP, Phase I entropies, Phase II gain-shares). All later sections reuse these exact models. Printed at the top of the report. | — |
| A | Provenance of CTI values: CVE publication dates vs the dataset snapshot dates, EPSS history start, the code lines that set the snapshot, per-service CTI values, and the same service compared across the two snapshot dates | R2.2, R3.2 |
| B | Port → service → CVE mapping: determinism, coverage, top (protocol/port → service) pairs for manual checking, availability by class | R3.3 |
| C | Port controls (numeric / categorical), a model with **only the information whether CTI context exists** (missing-value indicators + context_available flag), missingness structure | R1.1, R2.1, R2.5, R3.1 |
| D | Training randomness: is the frozen configuration deterministic; 5 seeds of a stochastic variant (bagging/feature fraction 0.8) | R1.4, R2.4, R3.4 |
| E | AP per attack class (class vs benign) for F0, F1, F0+port | R1.2 |
| F | UNSW-NB15 with alternative chronological boundaries (0.60/0.80, 0.80/0.90) | R1.3, R3.4 |
| D2 | *(added after the first run)* frozen configuration with 6 more seeds — only LightGBM's histogram-sampling seed changes | R1.4, R2.4, R3.4 |
| H | *(added after the first run)* CIC-IDS2017: F1 without the retrospective EPSS fields (EPSS taken from the 2021-04-14 file for a 2017 snapshot) | R2.2, R3.2 |
| G | *(added after the first run)* UNSW-NB15: train on all earlier days, test on the whole last capture day | R1.3, R3.4 |

Nothing in the input data is modified.

In [1]:
# ======================================================================================
# Round-1 revision checks — one run, one report
# ======================================================================================
import os, gc, re, json, time, glob, warnings
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score
warnings.filterwarnings("ignore")

# ---------------- configuration ----------------
REL = os.path.expanduser("~/notebooks/2 article/release")
OUT = os.path.join(REL, "audit", "results", "revision_r1")
CACHE = os.path.join(OUT, "cache")
UNSW_PATH = "/srv/ids_research/data/enriched/unsw_nb15_with_cve_context_dedup.parquet"
CIC_PATH = "/srv/ids_research/data/enriched/cic_ids2017_with_cve_context.parquet"
CVE_MASTER = "/srv/ids_research/data/cti/cve_master.parquet"
CTI_DIR = "/srv/ids_research/data/cti"
CODE_ROOTS = [os.path.expanduser("~/notebooks"), "/srv/ids_research"]
SNAP = {"UNSW-NB15": "2015-01-22", "CIC-IDS2017": "2017-07-03"}   # dataset-level CTI snapshot dates (Section 5.2)
EPSS_START = "2021-04-14"                                         # first day of EPSS history
SEED, ROUNDS, N_BOOT, TOP_PORTS = 42, 500, 500, 200
SEEDS_STOCH = [1, 2, 3, 4, 5]
STOCH = dict(bagging_fraction=0.8, bagging_freq=1, feature_fraction=0.8)
UNSW_ALT_SPLITS = [(0.60, 0.80), (0.80, 0.90)]                    # the paper's split is (0.70, 0.85)

F0_UNSW = ['state','dur','sbytes','dbytes','sttl','dttl','sloss','dloss','Sload','Dload','Spkts','Dpkts',
'swin','dwin','stcpb','dtcpb','smeansz','dmeansz','trans_depth','res_bdy_len','Sjit','Djit','Sintpkt',
'Dintpkt','tcprtt','synack','ackdat','is_sm_ips_ports','ct_state_ttl','ct_flw_http_mthd','is_ftp_login',
'ct_ftp_cmd','ct_srv_src','ct_srv_dst','ct_dst_ltm','ct_src_ltm','ct_src_dport_ltm','ct_dst_sport_ltm',
'ct_dst_src_ltm','proto','service']
CTX_UNSW = ['cve_ctx_service','cve_ctx_pool_size','cve_ctx_epss_coverage',
            'cve_ctx_n_generic_os_matches','cve_ctx_n_specific_product_matches']
F0_CIC = ['Flow Duration','Total Fwd Packets','Total Backward Packets','Total Length of Fwd Packets',
'Total Length of Bwd Packets','Fwd Packet Length Max','Fwd Packet Length Min','Fwd Packet Length Mean',
'Fwd Packet Length Std','Bwd Packet Length Max','Bwd Packet Length Min','Bwd Packet Length Mean',
'Bwd Packet Length Std','Flow Bytes/s','Flow Packets/s','Flow IAT Mean','Flow IAT Std','Flow IAT Max',
'Flow IAT Min','Fwd IAT Total','Fwd IAT Mean','Fwd IAT Std','Fwd IAT Max','Fwd IAT Min','Bwd IAT Total',
'Bwd IAT Mean','Bwd IAT Std','Bwd IAT Max','Bwd IAT Min','Fwd PSH Flags','Bwd PSH Flags','Fwd URG Flags',
'Bwd URG Flags','Fwd Header Length','Bwd Header Length','Fwd Packets/s','Bwd Packets/s','Min Packet Length',
'Max Packet Length','Packet Length Mean','Packet Length Std','Packet Length Variance','FIN Flag Count',
'SYN Flag Count','RST Flag Count','PSH Flag Count','ACK Flag Count','URG Flag Count','CWE Flag Count',
'ECE Flag Count','Down/Up Ratio','Average Packet Size','Avg Fwd Segment Size','Avg Bwd Segment Size',
'Fwd Avg Bytes/Bulk','Fwd Avg Packets/Bulk','Fwd Avg Bulk Rate','Bwd Avg Bytes/Bulk','Bwd Avg Packets/Bulk',
'Bwd Avg Bulk Rate','Subflow Fwd Packets','Subflow Fwd Bytes','Subflow Bwd Packets','Subflow Bwd Bytes',
'Init_Win_bytes_forward','Init_Win_bytes_backward','act_data_pkt_fwd','min_seg_size_forward','Active Mean',
'Active Std','Active Max','Active Min','Idle Mean','Idle Std','Idle Max','Idle Min','protocol']
CTX_CIC = ['cve_ctx_service','cve_ctx_pool_size','cve_ctx_n_epss','cve_ctx_epss_coverage',
           'cve_ctx_service_max_epss','cve_ctx_n_generic_os_matches','cve_ctx_n_specific_product_matches']

FROZEN = {
 "UNSW-NB15": dict(
    sizes=(1441568, 308915, 308935), spw=28.2135, attack_share=0.0821,
    ap=dict(F0=0.982439, F1=0.983854, Pnum=0.985116, Pcat=0.985920),
    auc=dict(F0=0.998144, F1=0.998123, Pnum=0.998132, Pcat=0.998390),
    rec=dict(F0=0.8028, F1=0.8218, Pnum=0.8414, Pcat=0.8490),
    # Table 10 / Table 13 CIs (unsw_port_cti_controls.ipynb, contrasts in this order, one RNG stream)
    ci_seq=[("F1 − F0", "F1", "F0", 0.001415, 0.000841, 0.002120),
            ("Pnum − F0", "Pnum", "F0", 0.002678, 0.001733, 0.003812),
            ("F1 − Pnum", "F1", "Pnum", -0.001262, -0.002097, -0.000625)],
    # Experiment B, S5 (expB_cti_beyond_port.ipynb, shared bootstrap samples)
    expB_S5=(-0.001262, -0.002121, -0.000510),
    per_class={"Exploits": 0.001552, "Shellcode": -0.033535, "Generic": 0.001250, "Reconnaissance": 0.011259,
               "DoS": 0.008956, "Fuzzers": 0.026415, "Analysis": 0.007749, "Backdoor": 0.007389, "Worms": 0.020737},
    phase1=[("cve_ctx_service", "native", 1.9093, 2.1454, 0.6542), ("cve_ctx_service", "proto+port", 9.1445, 2.1454, 0.0),
            ("all CTI", "proto+port", 9.1445, 2.1454, 0.0)],
    gain=dict(total=0.24, service=0.21, f0_top=("sttl", 96.24))),
 "CIC-IDS2017": dict(
    sizes=(1953816, 391055, 482806), spw=4.3357, attack_share=0.2923,
    ap=dict(F0=0.998869, F1=0.999402, Pnum=0.999489, Pcat=0.999562),
    auc=dict(F0=0.999402, F1=0.999638, Pnum=0.999754, Pcat=0.999800),
    rec=dict(F0=0.9777, F1=0.9934, Pnum=0.9925, Pcat=0.9942),
    # Table 10 / Table 13 CIs (cic_stratified_temporal.ipynb, contrasts in this order, one RNG stream)
    ci_seq=[("F1 − F0", "F1", "F0", 0.000533, 0.000183, 0.001988),
            ("F1 − Pnum", "F1", "Pnum", -0.000086, -0.000309, 0.000041),
            ("Pnum − F0", "Pnum", "F0", 0.000619, 0.000230, 0.002030)],
    expB_S5=(-0.000086, -0.000302, 0.000036),
    per_class={"DoS Slowhttptest": 0.179481, "DoS slowloris": 0.082810, "Web Attack – Brute Force": 0.050032,
               "SSH-Patator": 0.031811, "PortScan": -0.000112, "DoS Hulk": 0.001476, "DDoS": 0.000588,
               "DoS GoldenEye": 0.007244, "FTP-Patator": 0.001519, "Web Attack – XSS": 0.013861},
    phase1=[("cve_ctx_service", "native", 0.9548, 2.1829, 1.2556), ("cve_ctx_service", "proto+port", 5.1899, 2.1829, 0.0),
            ("all CTI", "proto+port", 5.1899, 2.1829, 0.0)],
    gain=dict(total=58.69, service=55.58, f0_top=("Bwd Packet Length Min", 25.50))),
}
VERIFY = []          # (dataset, item, expected, obtained, tolerance)
def V(ds, item, exp, got, tol):
    VERIFY.append((ds, item, exp, got, tol))

os.makedirs(CACHE, exist_ok=True)
LOG = os.path.join(OUT, "run_log.txt")
def _log(msg):
    with open(LOG, "a", encoding="utf-8") as fh:
        fh.write(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}\n")
try:
    from IPython.display import display, Pretty
    _h = display(Pretty("start…"), display_id=True)
    def progress(msg): _h.update(Pretty(f"[{time.strftime('%H:%M:%S')}] {msg}")); _log(msg)
except Exception:
    def progress(msg): print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True); _log(msg)

REPORT, ERRORS, RESULTS = [], [], {}
def R(line=""): REPORT.append(str(line))
def H(title): R(""); R("=" * 100); R(title); R("=" * 100)
def safe(n): return re.sub(r"[^0-9A-Za-z_]", "_", str(n))

def parse_port(v):
    s = str(v).strip().lower()
    try: return float(int(s, 16)) if s.startswith("0x") else float(s)
    except ValueError: return np.nan

# ---------------- loading ----------------
def load_unsw():
    df = pd.read_parquet(UNSW_PATH).reset_index(drop=True); df.columns = [c.strip() for c in df.columns]
    df["ct_ftp_cmd"] = pd.to_numeric(df["ct_ftp_cmd"].replace(" ", np.nan), errors="coerce")
    y = pd.to_numeric(df["Label"], errors="coerce").astype(int).to_numpy()
    t = pd.to_numeric(df["Stime"]).to_numpy()
    raw = df["dsport"].astype(str)
    port = raw.map({u: parse_port(u) for u in raw.unique()}).astype(float).to_numpy()
    cls = df["attack_cat"].astype(str).str.strip().replace({"nan": "Normal", "": "Normal"}).to_numpy()
    cls = np.where(y == 0, "Normal", cls)
    return dict(name="UNSW-NB15", df=df, y=y, t=t, ts=pd.Series(pd.to_datetime(t, unit="s")), block=(t // 60).astype(np.int64),
                port=port, cls=cls,
                benign="Normal", proto="proto", F0=F0_UNSW, CTX=CTX_UNSW, CAT={"state", "proto", "service"})

def load_cic():
    df = pd.read_parquet(CIC_PATH).reset_index(drop=True); df.columns = [c.strip() for c in df.columns]
    y = (df["attack_label"].astype(str) != "BENIGN").astype(int).to_numpy()
    raw = df["Timestamp"]
    ts = pd.to_datetime(raw) if np.issubdtype(raw.dtype, np.datetime64) else pd.to_datetime(raw.astype(str), dayfirst=True, errors="coerce")
    ts = ts.where(ts.dt.hour >= 8, ts + pd.Timedelta(hours=12))             # AM/PM correction
    port = pd.to_numeric(df["destination_port"], errors="coerce").astype(float).to_numpy()
    block = pd.factorize(ts.dt.floor("min"))[0].astype(np.int64)                 # as in Experiment B / phase12
    secs = ((ts - pd.Timestamp("1970-01-01")) // pd.Timedelta(seconds=1)).to_numpy()
    return dict(name="CIC-IDS2017", df=df, y=y, t=secs, ts=ts, block=block, port=port,
                cls=df["attack_label"].astype(str).to_numpy(), benign="BENIGN", proto="protocol",
                F0=F0_CIC, CTX=CTX_CIC, CAT={"protocol"})

def split_chrono(D, q1, q2):
    t = pd.Series(D["t"])
    b1, b2 = t.quantile(q1, interpolation="nearest"), t.quantile(q2, interpolation="nearest")
    return np.where(t < b1, 0, np.where(t < b2, 1, 2))

def split_strat(D):
    ts = pd.Series(pd.to_datetime(D["ts"])).reset_index(drop=True)
    grp = ts.dt.date.astype(str) + "|" + pd.Series(D["cls"]).astype(str)
    tn = ts.astype("int64")
    b1 = tn.groupby(grp).transform(lambda s: s.quantile(0.70, interpolation="nearest"))
    b2 = tn.groupby(grp).transform(lambda s: s.quantile(0.85, interpolation="nearest"))
    return np.where(tn < b1, 0, np.where(tn < b2, 1, 2))

def base_split(D):
    return split_chrono(D, 0.70, 0.85) if D["name"] == "UNSW-NB15" else split_strat(D)

# ---------------- features ----------------
def num_ctx(D): return [c for c in D["CTX"] if c != "cve_ctx_service"]

def avail_flag(D):
    df = D["df"]
    if "cve_ctx_context_available" in df.columns:
        return df["cve_ctx_context_available"].astype(str).str.lower().isin(["true", "1", "1.0"]).to_numpy()
    return ~df["cve_ctx_service"].astype(str).str.lower().isin(["nan", "none", "<na>", ""]).to_numpy()

def make_X(D, split):
    """F0 + CTI + port (numeric, categorical top-200 train ports) + CTI-missingness indicators; categories fitted on train."""
    df, tr, port = D["df"], split == 0, D["port"]
    X = df[D["F0"] + D["CTX"]].reset_index(drop=True).copy()
    X["port_num"] = port
    top = pd.Series(port[tr]).dropna().value_counts().index[:TOP_PORTS].to_numpy()
    pc = np.full(len(df), "other", dtype=object); m = np.isin(port, top)
    pc[m] = port[m].astype(np.int64).astype(str)
    X["port_cat"] = pc
    for c in num_ctx(D):
        X["miss_" + safe(c)] = pd.to_numeric(df[c], errors="coerce").isna().astype("int8").to_numpy()
    X["ctx_available"] = avail_flag(D).astype("int8")
    cats = set(D["CAT"]) | {"cve_ctx_service", "port_cat"}
    for c in cats:
        X[c] = pd.Categorical(X[c], categories=pd.Categorical(X.loc[tr, c]).categories)
    return X, cats

def model_sets(D):
    miss = ["miss_" + safe(c) for c in num_ctx(D)]
    return {"F0": D["F0"], "F1": D["F0"] + D["CTX"], "Pnum": D["F0"] + ["port_num"],
            "Pcat": D["F0"] + ["port_cat"], "AVAIL": D["F0"] + miss + ["ctx_available"]}

def fit_predict(D, X, cats, cols, split, tag, seed=SEED, stoch=False, save_gain=False):
    tr, te = split == 0, split == 2
    f = os.path.join(CACHE, f"{safe(D['name'])}_{safe(tag)}_{int(te.sum())}_{len(cols)}_s{seed}{'_st' if stoch else ''}.npy")
    fg = f.replace(".npy", "_gain.json")
    if os.path.exists(f) and (not save_gain or os.path.exists(fg)):
        return np.load(f)
    y = D["y"]
    params = dict(objective="binary", metric="None", seed=seed, num_leaves=15, learning_rate=0.05,
                  min_data_in_leaf=50, verbosity=-1, scale_pos_weight=(y[tr] == 0).sum() / y[tr].sum())
    if stoch: params.update(STOCH)
    cc = [c for c in cols if c in cats]
    m = lgb.train(params, lgb.Dataset(X.loc[tr, cols], label=y[tr], categorical_feature=cc), num_boost_round=ROUNDS)
    p = m.predict(X.loc[te, cols]).astype("float64")
    if save_gain:
        g = pd.Series(m.feature_importance(importance_type="gain"), index=cols)
        json.dump((100 * g / g.sum()).to_dict(), open(fg, "w"))
    np.save(f, p); del m; gc.collect()
    return p

def boot_samples(D, split):
    te = split == 2
    _, inv = np.unique(D["block"][te], return_inverse=True)
    blocks = list(pd.Series(np.arange(len(inv))).groupby(inv).indices.values())
    rng = np.random.default_rng(SEED)
    return [np.concatenate([blocks[k] for k in rng.integers(0, len(blocks), len(blocks))]) for _ in range(N_BOOT)]

def contrast(yt, pa, pb, samples):
    d = average_precision_score(yt, pa) - average_precision_score(yt, pb)
    v = np.array([average_precision_score(yt[j], pa[j]) - average_precision_score(yt[j], pb[j]) for j in samples])
    lo, hi = np.percentile(v, [2.5, 97.5])
    return dict(d=float(d), lo=float(lo), hi=float(hi), p=float((1 + np.sum(v <= 0)) / (1 + len(v))))

from sklearn.metrics import roc_curve
def recall_at_fpr(yt, p, fpr_t=0.001):
    fpr, tpr, _ = roc_curve(yt, p); ok = fpr <= fpr_t
    return float(tpr[ok].max()) if ok.any() else 0.0

def entropy_bits(codes):
    _, cnt = np.unique(codes, return_counts=True); p = cnt / cnt.sum()
    return float(-(p * np.log2(p)).sum())

def joint_codes(*cols):
    code = None
    for c in cols:
        k = pd.factorize(pd.Series(c), use_na_sentinel=False)[0].astype(np.int64)
        code = k if code is None else pd.factorize(code * (int(k.max()) + 1) + k)[0].astype(np.int64)
    return code

def overlap(C, F):
    HC, HF = entropy_bits(C), entropy_bits(F)
    return HF, HC, max(entropy_bits(joint_codes(C, F)) - HF, 0.0)

def legacy_seq_ci(D, split, P, seq):
    """Re-create the bootstrap of the source notebooks: blocks in their order, one RNG stream consumed contrast by contrast."""
    te = split == 2; yt = D["y"][te]
    if D["name"] == "UNSW-NB15":
        key = (D["t"][te] // 60).astype(np.int64)
    else:
        key = pd.factorize(pd.Series(pd.to_datetime(D["ts"])).reset_index(drop=True)[te].dt.floor("min"))[0]
    blocks = list(pd.Series(np.arange(len(key))).groupby(key).indices.values())
    rng = np.random.default_rng(SEED); out = []
    for lab, a, b, *_ in seq:
        pa, pb = P[a], P[b]
        d = average_precision_score(yt, pa) - average_precision_score(yt, pb); vals = []
        for _ in range(N_BOOT):
            j = np.concatenate([blocks[k] for k in rng.integers(0, len(blocks), len(blocks))])
            vals.append(average_precision_score(yt[j], pa[j]) - average_precision_score(yt[j], pb[j]))
        lo, hi = np.percentile(vals, [2.5, 97.5]); out.append((d, lo, hi))
    return out

def fmt_c(c): return f"{c['d']:+.6f}  [{c['lo']:+.6f}, {c['hi']:+.6f}]  p={c['p']:.4f}"

# ---------------- A. provenance of CTI values (EPSS / CVE dates) ----------------
def section_A(datasets):
    H("A. PROVENANCE OF THE CTI VALUES (reviewers 2.2, 3.2)")
    # A1: CTI corpus files
    R("A1. Files in the CTI folder:")
    for f in sorted(glob.glob(os.path.join(CTI_DIR, "**", "*"), recursive=True)):
        if os.path.isfile(f):
            R(f"   {os.path.relpath(f, CTI_DIR):<60} {os.path.getsize(f) / 1e6:9.1f} MB")
    # A2: cve_master schema and dates
    R(""); R(f"A2. {CVE_MASTER}")
    cm = pd.read_parquet(CVE_MASTER)
    R(f"   rows = {len(cm):,}; columns = {len(cm.columns)}")
    for c in cm.columns:
        s = cm[c]; line = f"   {c:<40} {str(s.dtype):<16} non-null={s.notna().mean():6.1%}"
        if pd.api.types.is_datetime64_any_dtype(s.dtype) or (not pd.api.types.is_numeric_dtype(s) and re.search(r"date|publish|time|modified", c, re.I)):
            d = pd.to_datetime(s, errors="coerce", utc=True)
            if d.notna().any():
                line += f"  min={d.min().date()}  max={d.max().date()}"
        elif pd.api.types.is_numeric_dtype(s) and s.notna().any():
            line += f"  min={s.min():.4g}  max={s.max():.4g}"
        R(line)
    pub_cols = [c for c in cm.columns if re.search(r"publish", c, re.I)]
    if pub_cols:
        pub = pd.to_datetime(cm[pub_cols[0]], errors="coerce", utc=True)
        R(f"   publication column used: {pub_cols[0]}")
        for ds, snap in SNAP.items():
            s = pd.Timestamp(snap, tz="UTC")
            R(f"   CVEs published AFTER the {ds} snapshot ({snap}): {int((pub > s).sum()):,} "
              f"({(pub > s).mean():.1%} of the corpus)")
        R(f"   CVEs published before EPSS history began ({EPSS_START}): {int((pub < pd.Timestamp(EPSS_START, tz='UTC')).sum()):,}")
    del cm; gc.collect()
    # A3: code that sets the snapshot / EPSS fields
    R(""); R("A3. Code lines that define the snapshot and the EPSS/pool fields (notebooks and scripts on the server):")
    pat = re.compile(r"t_snap|snapshot|snap_date|SNAP|service_max_epss|n_epss|epss_coverage|pool_size|2017-07-03|2015-01-22|published", re.I)
    shown = 0
    for root in CODE_ROOTS:
        cands = glob.glob(os.path.join(root, "**", "*.ipynb"), recursive=True) + glob.glob(os.path.join(root, "**", "*.py"), recursive=True)
        for f in sorted(cands):
            if shown >= 15: break
            if any(x in f for x in (".ipynb_checkpoints", "revision_r1", "/audit/", "site-packages", "/envs/", "/.cache/")):
                continue
            try:
                if os.path.getsize(f) > 8e6: continue
                if f.endswith(".ipynb"):
                    nb = json.load(open(f, encoding="utf-8"))
                    lines = [l for c in nb.get("cells", []) if c.get("cell_type") == "code"
                             for l in "".join(c.get("source", "")).splitlines()]
                else:
                    lines = open(f, encoding="utf-8", errors="ignore").read().splitlines()
            except Exception:
                continue
            text = "\n".join(lines)
            if "cve_ctx" not in text or not re.search(r"epss", text, re.I):
                continue
            hits = [(i, l) for i, l in enumerate(lines) if pat.search(l)]
            if not hits: continue
            shown += 1
            R(f"   --- {f}  ({len(hits)} matching lines)")
            for i, l in hits[:30]:
                R(f"      {i:>5}: {l.strip()[:160]}")
    if shown == 0:
        R("   no notebook/script containing both 'cve_ctx' and 'epss' was found under " + ", ".join(CODE_ROOTS))
    # A4: per-service CTI values in each enriched dataset
    for D in datasets:
        R(""); R(f"A4. {D['name']}: CTI values per service (snapshot {SNAP[D['name']]}); top 30 services by flow count")
        df = D["df"]; svc = df["cve_ctx_service"].astype(str)
        allc = [c for c in df.columns if c.startswith("cve_ctx_") and c != "cve_ctx_service"]
        g = df[allc].apply(pd.to_numeric, errors="coerce").groupby(svc)
        tab = g.first(); tab.insert(0, "flows", svc.value_counts()); tab.insert(1, "attack_share", pd.Series(D["y"]).groupby(svc.to_numpy()).mean())
        R("   " + tab.sort_values("flows", ascending=False).head(30).round(4).to_string().replace("\n", "\n   "))
        nun = g.nunique(dropna=False).max()
        R("   max distinct values within a service: " + ", ".join(f"{k}={int(v)}" for k, v in nun.items()))
        D["_svc_tab"] = tab
    if len(datasets) == 2:
        a, b = datasets[0], datasets[1]
        ta, tb = a["_svc_tab"], b["_svc_tab"]
        common = [s for s in ta.index if s in tb.index and s.lower() not in ("nan", "none", "<na>")]
        cols = [c for c in ["cve_ctx_pool_size", "cve_ctx_n_epss", "cve_ctx_service_max_epss", "cve_ctx_epss_coverage"] if c in ta.columns and c in tb.columns]
        R(""); R(f"A5. Same service, two snapshot dates ({a['name']} {SNAP[a['name']]} vs {b['name']} {SNAP[b['name']]}):")
        R("    identical values at both dates suggest that the CVE pool / EPSS were NOT restricted to the snapshot date")
        if common and cols:
            cmp = pd.concat({a["name"]: ta.loc[common, cols], b["name"]: tb.loc[common, cols]}, axis=1)
            R("   " + cmp.round(4).to_string().replace("\n", "\n   "))
            for c in cols:
                same = int(np.isclose(ta.loc[common, c].astype(float), tb.loc[common, c].astype(float), equal_nan=True).sum())
                R(f"   {c:<28} identical in {same}/{len(common)} common services")

# ---------------- B. port -> service -> CVE mapping ----------------
def section_B(D):
    H(f"B. {D['name']}: PORT -> SERVICE -> CVE MAPPING (reviewer 3.3)")
    df, y = D["df"], D["y"]
    svc = df["cve_ctx_service"].astype(str).to_numpy()
    proto = df[D["proto"]].astype(str).to_numpy()
    port = pd.Series(D["port"]).fillna(-1).astype(np.int64).astype(str).to_numpy()
    avail = avail_flag(D)
    R(f"flows = {len(df):,}; distinct (protocol, port) pairs = {pd.Series(proto + '/' + port).nunique():,}; services = {pd.Series(svc).nunique()}")
    R(f"CTI context available: all {avail.mean():.2%} | benign {avail[y == 0].mean():.2%} | attack {avail[y == 1].mean():.2%}")
    t = pd.DataFrame({"pp": pd.Series(proto) + "/" + pd.Series(port), "svc": svc, "y": y, "avail": avail})
    one = t.groupby("pp")["svc"].nunique()
    R(f"(protocol, port) pairs mapped to more than one service: {int((one > 1).sum())}  (0 = deterministic mapping)")
    top = t.groupby(["pp", "svc"]).agg(flows=("y", "size"), attack_share=("y", "mean"), ctx=("avail", "mean"))
    top = top.sort_values("flows", ascending=False).head(25).reset_index()
    pool = df.groupby(df["cve_ctx_service"].astype(str))["cve_ctx_pool_size"].first()
    top["pool_size"] = top["svc"].map(pool)
    R("top 25 (protocol/port -> service) pairs by flow count, for manual checking:")
    R("   " + top.round(4).to_string(index=False).replace("\n", "\n   "))
    cls = pd.Series(D["cls"])
    by = pd.DataFrame({"cls": cls, "avail": avail}).groupby("cls")["avail"].agg(["size", "mean"]).sort_values("size", ascending=False)
    R("CTI context availability by class:")
    R("   " + by.rename(columns={"size": "flows", "mean": "available"}).round(4).to_string().replace("\n", "\n   "))
    RESULTS.setdefault(D["name"], {})["mapping"] = dict(avail_all=float(avail.mean()), avail_benign=float(avail[y == 0].mean()),
                                                         avail_attack=float(avail[y == 1].mean()), multi_service_pairs=int((one > 1).sum()))

# ---------------- 0. verification of the frozen results of the paper ----------------
def verify_frozen(D, split, P, sets):
    name, y, Fz = D["name"], D["y"], FROZEN[D["name"]]
    tr, va, te = split == 0, split == 1, split == 2; yt = y[te]
    progress(f"{name}: verifying frozen results")
    for i, part in enumerate(["train", "val", "test"]):
        V(name, f"split size ({part})", Fz["sizes"][i], int((split == i).sum()), 0)
    V(name, "scale_pos_weight", Fz["spw"], float((y[tr] == 0).sum() / y[tr].sum()), 5e-5)
    V(name, "attack share in test", Fz["attack_share"], float(yt.mean()), 5e-5)
    for k in ["F0", "F1", "Pnum", "Pcat"]:
        V(name, f"AP {k} (Tables 10, 13, 16)", Fz["ap"][k], float(average_precision_score(yt, P[k])), 5e-7)
        V(name, f"ROC-AUC {k} (Table 16)", Fz["auc"][k], float(roc_auc_score(yt, P[k])), 5e-7)
        V(name, f"recall@FPR0.1% {k} (Tables 10, 16)", Fz["rec"][k], recall_at_fpr(yt, P[k]), 5e-5)
    for (lab, a, b, ed, elo, ehi), (d, lo, hi) in zip(Fz["ci_seq"], legacy_seq_ci(D, split, P, Fz["ci_seq"])):
        V(name, f"ΔAP {lab} (Tables 10/13)", ed, float(d), 5e-7)
        V(name, f"CI low {lab}", elo, float(lo), 5e-7); V(name, f"CI high {lab}", ehi, float(hi), 5e-7)
    c = contrast(yt, P["F1"], P["Pnum"], boot_samples(D, split))
    for nm, e, g in zip(["ΔAP", "CI low", "CI high"], Fz["expB_S5"], [c["d"], c["lo"], c["hi"]]):
        V(name, f"{nm} F1 − Pnum, Experiment B S5 (Table 17)", e, g, 5e-7)
    cls_te = D["cls"][te]; ben = cls_te == D["benign"]
    for cl, e in Fz["per_class"].items():
        pos = cls_te == cl
        if pos.sum() == 0: V(name, f"per-class ΔAP {cl}", e, float("nan"), 5e-7); continue
        msk = pos | ben; yy = pos[msk].astype(int)
        V(name, f"per-class ΔAP F1−F0 {cl} (Section 5.2)", e,
          float(average_precision_score(yy, P["F1"][msk]) - average_precision_score(yy, P["F0"][msk])), 5e-7)
    # Phase I (Table 11), training partition only, exactly as phase12_corrected.ipynb
    df = D["df"]; svc = df["cve_ctx_service"].astype(str).to_numpy()
    proto = df[D["proto"]].astype(str).to_numpy(); port_s = pd.Series(D["port"]).astype(str).to_numpy()
    native = df["service" if name.startswith("UNSW") else "protocol"].astype(str).to_numpy()
    Csvc = joint_codes(svc[tr]); Call = joint_codes(*[df[c].to_numpy()[tr] for c in D["CTX"]])
    Fpp = joint_codes(proto[tr], port_s[tr])
    got = [overlap(Csvc, joint_codes(native[tr])), overlap(Csvc, Fpp), overlap(Call, Fpp)]
    for (cn, fn, eHF, eHC, eHCF), (HF, HC, HCF) in zip(Fz["phase1"], got):
        V(name, f"Phase I H(F) {cn} | {fn} (Table 11)", eHF, HF, 5e-5)
        V(name, f"Phase I H(C) {cn} | {fn}", eHC, HC, 5e-5)
        V(name, f"Phase I H(C|F) {cn} | {fn}", eHCF, HCF, 5e-5)
    # Phase II gain-share (Table 12)
    te_n = int(te.sum())
    g1 = json.load(open(os.path.join(CACHE, f"{safe(name)}_F1_{te_n}_{len(sets['F1'])}_s{SEED}_gain.json")))
    g0 = json.load(open(os.path.join(CACHE, f"{safe(name)}_F0_{te_n}_{len(sets['F0'])}_s{SEED}_gain.json")))
    V(name, "Phase II gain-share, all CTI in F1 (%) (Table 12)", Fz["gain"]["total"], sum(g1[c] for c in D["CTX"]), 0.005)
    V(name, "Phase II gain-share, cve_ctx_service in F1 (%)", Fz["gain"]["service"], g1["cve_ctx_service"], 0.005)
    f, e = Fz["gain"]["f0_top"]
    V(name, f"Phase II gain-share, {f} in F0 (%)", e, g0.get(f, float("nan")), 0.005)

def report_verification():
    H("0. VERIFICATION OF THE FROZEN RESULTS OF THE PAPER (all later sections reuse these exact models)")
    bad = 0
    for ds, item, e, g, tol in VERIFY:
        ok = (g == g) and abs(float(g) - float(e)) <= tol
        bad += not ok
        fe = f"{e:,}" if isinstance(e, int) else f"{e:.6f}"
        fg = f"{g:,}" if isinstance(g, (int, np.integer)) else f"{g:.6f}"
        R(f"   {'OK      ' if ok else 'MISMATCH'} {ds:<12} {item:<58} paper={fe:>14}  now={fg:>14}")
    R(""); R(f"   SUMMARY: {len(VERIFY) - bad}/{len(VERIFY)} frozen values reproduced"
             + ("" if bad == 0 else f";  {bad} MISMATCH — read the later sections only after resolving them"))
    RESULTS["verification"] = dict(checked=len(VERIFY), mismatches=bad)
    return bad

# ---------------- C-F. models ----------------
def section_models(D):
    name, y = D["name"], D["y"]
    split = base_split(D); tr, te = split == 0, split == 2; yt = y[te]
    progress(f"{name}: features"); X, cats = make_X(D, split)
    sets = model_sets(D)
    P = {}
    for i, (k, cols) in enumerate(sets.items(), 1):
        progress(f"{name}: base model {k} ({i}/{len(sets)})")
        P[k] = fit_predict(D, X, cats, cols, split, k, save_gain=k in ("F0", "F1"))
    ap = {k: average_precision_score(yt, p) for k, p in P.items()}
    verify_frozen(D, split, P, sets)
    progress(f"{name}: bootstrap"); S = boot_samples(D, split)
    H(f"C. {name}: BASE MODELS (T={ROUNDS}, frozen configuration, seed {SEED}) — port and CTI-availability controls")
    for k in ["F0", "F1", "Pnum", "Pcat"]:
        R(f"   {k:<5} AP={ap[k]:.6f}   (verified against the paper in section 0)")
    R(f"   AVAIL AP={ap['AVAIL']:.6f}   (F0 + only the information whether CTI context exists: missing-value indicators + context_available flag)")
    cs = {}
    for lab, a, b in [("F1 − F0", "F1", "F0"), ("F1 − Pnum", "F1", "Pnum"), ("F1 − Pcat", "F1", "Pcat"),
                      ("AVAIL − F0", "AVAIL", "F0"), ("Pcat − AVAIL", "Pcat", "AVAIL")]:
        cs[lab] = contrast(yt, P[a], P[b], S); R(f"   {lab:<12} {fmt_c(cs[lab])}")
    # missingness structure (reviewer 2.5)
    R(""); R("   CTI missingness (reviewer 2.5):")
    svc = D["df"]["cve_ctx_service"].astype(str)
    for c in num_ctx(D):
        miss = pd.to_numeric(D["df"][c], errors="coerce").isna()
        within = miss.groupby(svc.to_numpy()).nunique().max()
        R(f"   {c:<38} missing: all {miss.mean():6.2%}  benign {miss[y == 0].mean():6.2%}  attack {miss[y == 1].mean():6.2%}"
          f"   distinct missing-states within a service: {int(within)}")
    RESULTS.setdefault(name, {})["base"] = dict(ap={k: float(v) for k, v in ap.items()}, contrasts=cs)

    # D. seeds
    H(f"D. {name}: TRAINING RANDOMNESS (reviewers 1.4, 2.4, 3.4)")
    p7 = fit_predict(D, X, cats, sets["F0"], split, "F0", seed=7)
    diff = float(np.max(np.abs(p7 - P["F0"])))
    R(f"   frozen configuration, F0 with seed 7 vs seed {SEED}: max |difference in scores| = {diff:.3e} "
      f"({'deterministic: the seed has no effect' if diff < 1e-12 else 'NOT deterministic'})")
    R(f"   stochastic variant ({', '.join(f'{k}={v}' for k, v in STOCH.items())}), seeds {SEEDS_STOCH}:")
    rows = []
    for s in SEEDS_STOCH:
        row = {"seed": s}
        for k in ["F0", "F1", "Pnum", "Pcat"]:
            progress(f"{name}: seed {s}, {k}")
            row[k] = average_precision_score(yt, fit_predict(D, X, cats, sets[k], split, k, seed=s, stoch=True))
        rows.append(row)
    sd = pd.DataFrame(rows).set_index("seed")
    sd["F1−F0"] = sd["F1"] - sd["F0"]; sd["F1−Pnum"] = sd["F1"] - sd["Pnum"]; sd["F1−Pcat"] = sd["F1"] - sd["Pcat"]
    R("   " + sd.round(6).to_string().replace("\n", "\n   "))
    R(f"   mean ± sd:  " + "  ".join(f"{c}={sd[c].mean():+.6f}±{sd[c].std(ddof=1):.6f}" for c in sd.columns))
    R(f"   seeds with F1−F0 > 0: {int((sd['F1−F0'] > 0).sum())}/{len(sd)};  F1−Pnum > 0: {int((sd['F1−Pnum'] > 0).sum())}/{len(sd)};"
      f"  F1−Pcat > 0: {int((sd['F1−Pcat'] > 0).sum())}/{len(sd)}")
    RESULTS[name]["seeds"] = dict(frozen_max_diff=diff, table=sd.reset_index().to_dict(orient="list"))

    # E. per-class AP
    H(f"E. {name}: AP PER ATTACK CLASS (class vs benign, T={ROUNDS}) (reviewer 1.2)")
    cls_te = D["cls"][te]; ben = cls_te == D["benign"]
    rows = []
    for c in sorted(set(cls_te) - {D["benign"]}):
        pos = cls_te == c; msk = pos | ben; yy = pos[msk].astype(int)
        r = {"class": c, "n_test": int(pos.sum())}
        for k in ["F0", "F1", "Pcat"]:
            r[k] = average_precision_score(yy, P[k][msk])
        r["F1−F0"] = r["F1"] - r["F0"]; r["F1−Pcat"] = r["F1"] - r["Pcat"]
        rows.append(r)
    pc = pd.DataFrame(rows).sort_values("n_test", ascending=False)
    R("   " + pc.round(6).to_string(index=False).replace("\n", "\n   "))
    R("   (classes with fewer than 50 test flows should be read descriptively only)")
    RESULTS[name]["per_class"] = pc.to_dict(orient="list")
    del X; gc.collect()

    # F. temporal robustness (UNSW only)
    if name == "UNSW-NB15":
        H("F. UNSW-NB15: ALTERNATIVE CHRONOLOGICAL BOUNDARIES (reviewers 1.3, 3.4)")
        R(f"   {'split':<11} {'test span':<38} {'hours':>6} {'attack%':>8} {'absent in train':<18} {'AP F0':>9} {'AP F1':>9} {'AP Pcat':>9}")
        alt = {}
        for q1, q2 in [(0.70, 0.85)] + UNSW_ALT_SPLITS:
            sp = split_chrono(D, q1, q2); te2 = sp == 2; yt2 = y[te2]
            if (q1, q2) == (0.70, 0.85):
                P2, X2 = P, None
            else:
                progress(f"UNSW: split {q1}/{q2}"); X2, cats2 = make_X(D, sp)
                P2 = {k: fit_predict(D, X2, cats2, sets[k], sp, f"{k}_q{q1}_{q2}") for k in ["F0", "F1", "Pcat"]}
            tt = D["ts"][te2]
            absent = sorted(set(D["cls"][te2]) - set(D["cls"][sp == 0]))
            a = {k: average_precision_score(yt2, P2[k]) for k in ["F0", "F1", "Pcat"]}
            R(f"   {q1:.2f}/{q2:.2f}   {str(tt.min())[:19]} – {str(tt.max())[:19]} {(tt.max() - tt.min()).total_seconds() / 3600:>6.1f}"
              f" {100 * yt2.mean():>7.2f}% {(','.join(absent) or '—'):<18} {a['F0']:>9.6f} {a['F1']:>9.6f} {a['Pcat']:>9.6f}")
            S2 = S if X2 is None else boot_samples(D, sp)
            c1 = contrast(yt2, P2["F1"], P2["F0"], S2); c2 = contrast(yt2, P2["F1"], P2["Pcat"], S2)
            R(f"              F1 − F0   {fmt_c(c1)}"); R(f"              F1 − Pcat {fmt_c(c2)}")
            alt[f"{q1}/{q2}"] = dict(ap=a, F1_F0=c1, F1_Pcat=c2, test_hours=(tt.max() - tt.min()).total_seconds() / 3600, absent=absent)
            if X2 is not None: del X2; gc.collect()
        RESULTS[name]["alt_splits"] = alt

# ---------------- checkpoints ----------------
SEC_DIR = os.path.join(OUT, "sections")
os.makedirs(SEC_DIR, exist_ok=True)

def save_outputs(final=False):
    """Write everything computed so far: report (verification first), JSON of all numbers."""
    lines = REPORT[:]; REPORT.clear()
    try:
        if VERIFY: report_verification()
    except Exception as e:
        R(f"(verification report failed: {e})")
    ver = REPORT[:]; REPORT.clear(); REPORT.extend(lines)
    tail = [] if final else ["", "(PARTIAL — the run has not finished; re-run the notebook to continue from the last finished section)"]
    name = "revision_r1_report.txt" if final else "revision_r1_report_PARTIAL.txt"
    open(os.path.join(OUT, name), "w", encoding="utf-8").write("\n".join(ver + lines + tail))
    json.dump(dict(RESULTS, verification_items=[list(v) for v in VERIFY]),
              open(os.path.join(OUT, "revision_r1_results.json" if final else "revision_r1_results_PARTIAL.json"), "w"),
              indent=1, default=str)

def run_section(key, fn, *args):
    """Run a section once; on a re-run, reuse its saved output instead of recomputing it."""
    f = os.path.join(SEC_DIR, safe(key) + ".json")
    if os.path.exists(f):
        d = json.load(open(f, encoding="utf-8"))
        REPORT.extend(d["lines"]); RESULTS.update(d["results"]); VERIFY.extend(tuple(v) for v in d["verify"])
        progress(f"section {key}: loaded from checkpoint"); return
    n_rep, n_ver, before = len(REPORT), len(VERIFY), json.dumps(RESULTS, default=str)
    progress(f"section {key}"); fn(*args)
    old = json.loads(before)
    new_res = {k: v for k, v in json.loads(json.dumps(RESULTS, default=str)).items() if old.get(k) != v}
    json.dump(dict(lines=REPORT[n_rep:], results=new_res, verify=[list(v) for v in VERIFY[n_ver:]]),
              open(f, "w", encoding="utf-8"), indent=1, default=str)
    save_outputs(final=False)

# ---------------- D2 / G. added after the first full run ----------------
SEEDS_FROZEN = [1, 2, 3, 4, 5, 7]

def section_extra(D):
    name, y = D["name"], D["y"]
    split = base_split(D); te = split == 2; yt = y[te]
    progress(f"{name}: features (extra)"); X, cats = make_X(D, split)
    sets = model_sets(D)
    base = {k: average_precision_score(yt, fit_predict(D, X, cats, sets[k], split, k, save_gain=k in ("F0", "F1")))
            for k in ["F0", "F1", "Pnum", "Pcat"]}
    H(f"D2. {name}: FROZEN CONFIGURATION, DIFFERENT SEEDS (reviewers 1.4, 2.4, 3.4)")
    R("   Only the random seed changes (it drives LightGBM's sampling of rows used to build the feature histograms);")
    R("   every other setting is the frozen configuration of the paper. Seed 42 = the paper.")
    rows = [dict(seed=42, **base)]
    for sd in SEEDS_FROZEN:
        row = {"seed": sd}
        for k in ["F0", "F1", "Pnum", "Pcat"]:
            progress(f"{name}: frozen config, seed {sd}, {k}")
            row[k] = average_precision_score(yt, fit_predict(D, X, cats, sets[k], split, k, seed=sd))
        rows.append(row)
    t = pd.DataFrame(rows).set_index("seed")
    t["F1−F0"] = t["F1"] - t["F0"]; t["F1−Pnum"] = t["F1"] - t["Pnum"]; t["F1−Pcat"] = t["F1"] - t["Pcat"]
    R("   " + t.round(6).to_string().replace("\n", "\n   "))
    R("   mean ± sd over all 7 seeds:")
    for c in t.columns:
        R(f"      {c:<8} {t[c].mean():+.6f} ± {t[c].std(ddof=1):.6f}   min {t[c].min():+.6f}   max {t[c].max():+.6f}")
    R(f"   seeds with F1−F0 > 0: {int((t['F1−F0'] > 0).sum())}/7;  F1−Pnum > 0: {int((t['F1−Pnum'] > 0).sum())}/7;"
      f"  F1−Pcat > 0: {int((t['F1−Pcat'] > 0).sum())}/7")
    RESULTS.setdefault(name, {})["seeds_frozen"] = t.reset_index().to_dict(orient="list")

    if name == "CIC-IDS2017":
        H("H. CIC-IDS2017: F1 WITHOUT THE RETROSPECTIVE EPSS FIELDS (reviewers 2.2, 3.2)")
        R("   For the 2017-07-03 snapshot the enrichment took EPSS from the earliest available file (2021-04-14),")
        R("   i.e. retrospectively. This model drops every EPSS-derived field from F1 and keeps the rest.")
        ep = [c for c in D["CTX"] if "epss" in c]
        cols = [c for c in sets["F1"] if c not in ep]
        R(f"   dropped: {ep}")
        p_ne = fit_predict(D, X, cats, cols, split, "F1noEPSS")
        P = {k: fit_predict(D, X, cats, sets[k], split, k) for k in ["F0", "F1", "Pcat"]}
        R(f"   F1noEPSS AP={average_precision_score(yt, p_ne):.6f}   (F1 {average_precision_score(yt, P['F1']):.6f})")
        S = boot_samples(D, split); h = {}
        for lab, pa, pb in [("F1 − F1noEPSS", P["F1"], p_ne), ("F1noEPSS − F0", p_ne, P["F0"]), ("F1noEPSS − Pcat", p_ne, P["Pcat"])]:
            h[lab] = contrast(yt, pa, pb, S); R(f"   {lab:<16} {fmt_c(h[lab])}")
        RESULTS[name]["no_epss"] = dict(dropped=ep, contrasts=h)

    if name == "UNSW-NB15":
        H("G. UNSW-NB15: TEST ON THE WHOLE LAST CAPTURE DAY (reviewers 1.3, 3.4)")
        day = pd.Series(D["ts"]).dt.normalize().to_numpy()
        last = day.max()
        days = pd.Series(pd.Series(D["ts"]).dt.date).value_counts().sort_index()
        R("   flows per capture day: " + ", ".join(f"{d}: {n:,}" for d, n in days.items()))
        sp = np.where(day < last, 0, 2)          # train = all earlier days, test = the whole last day (no validation needed)
        te2 = sp == 2; yt2 = y[te2]; tt = pd.Series(D["ts"])[te2]
        absent = sorted(set(D["cls"][te2]) - set(D["cls"][sp == 0]))
        R(f"   train = {int((sp == 0).sum()):,} flows (before {str(last)[:10]}), test = {int(te2.sum()):,} flows "
          f"({str(tt.min())[:19]} – {str(tt.max())[:19]}, {(tt.max() - tt.min()).total_seconds() / 3600:.1f} h), "
          f"attack share in test {100 * yt2.mean():.2f}%")
        R(f"   attack classes in test but absent from train: {', '.join(absent) or '—'}")
        X2, cats2 = make_X(D, sp)
        P2 = {k: fit_predict(D, X2, cats2, sets[k], sp, f"{k}_lastday") for k in ["F0", "F1", "Pnum", "Pcat"]}
        for k in P2:
            R(f"   {k:<5} AP={average_precision_score(yt2, P2[k]):.6f}")
        S2 = boot_samples(D, sp); g = {}
        for lab, a, b in [("F1 − F0", "F1", "F0"), ("F1 − Pnum", "F1", "Pnum"), ("F1 − Pcat", "F1", "Pcat")]:
            g[lab] = contrast(yt2, P2[a], P2[b], S2); R(f"   {lab:<10} {fmt_c(g[lab])}")
        if absent:
            keep = ~np.isin(D["cls"][te2], absent)
            R(f"   same, excluding test flows of the classes absent from train ({int((~keep).sum()):,} flows):")
            for lab, a, b in [("F1 − F0", "F1", "F0"), ("F1 − Pcat", "F1", "Pcat")]:
                d = average_precision_score(yt2[keep], P2[a][keep]) - average_precision_score(yt2[keep], P2[b][keep])
                R(f"      {lab:<10} ΔAP={d:+.6f}")
        RESULTS[name]["last_day"] = dict(contrasts=g, absent=absent, test_hours=(tt.max() - tt.min()).total_seconds() / 3600)
        del X2; gc.collect()
    del X; gc.collect()

# ---------------- run ----------------
t0 = time.time()
DS = []
for loader in (load_unsw, load_cic):
    try:
        progress(f"loading {loader.__name__}"); DS.append(loader())
    except Exception as e:
        import traceback; ERRORS.append(f"{loader.__name__}: {type(e).__name__}: {e}\n{traceback.format_exc()}")
for step, fn, args in [("A", section_A, (DS,))] + [(f"B {D['name']}", section_B, (D,)) for D in DS] \
                      + [(f"models {D['name']}", section_models, (D,)) for D in DS] \
                      + [(f"extra {D['name']}", section_extra, (D,)) for D in DS]:
    try:
        run_section(step, fn, *args)
    except Exception as e:
        import traceback; ERRORS.append(f"section {step}: {type(e).__name__}: {e}\n{traceback.format_exc()}")
R(""); R(f"Run time: {(time.time() - t0) / 60:.1f} min;  ROUNDS={ROUNDS}, N_BOOT={N_BOOT}, SEEDS_STOCH={SEEDS_STOCH}")
if ERRORS:
    R(""); R("ERRORS:"); [R(e) for e in ERRORS]
save_outputs(final=True)
for f in ("revision_r1_report_PARTIAL.txt", "revision_r1_results_PARTIAL.json"):
    if os.path.exists(os.path.join(OUT, f)): os.remove(os.path.join(OUT, f))
progress(f"done — report below (also saved to {OUT})")
print(open(os.path.join(OUT, "revision_r1_report.txt"), encoding="utf-8").read())


[07:36:37] done — report below (also saved to ~/notebooks/2 article/release/audit/results/revision_r1)


0. VERIFICATION OF THE FROZEN RESULTS OF THE PAPER (all later sections reuse these exact models)
   OK       UNSW-NB15    split size (train)                                         paper=     1,441,568  now=     1,441,568
   OK       UNSW-NB15    split size (val)                                           paper=       308,915  now=       308,915
   OK       UNSW-NB15    split size (test)                                          paper=       308,935  now=       308,935
   OK       UNSW-NB15    scale_pos_weight                                           paper=     28.213500  now=     28.213472
   OK       UNSW-NB15    attack share in test                                       paper=      0.082100  now=      0.082069
   OK       UNSW-NB15    AP F0 (Tables 10, 13, 16)                                  paper=      0.982439  now=      0.982439
   OK       UNSW-NB15    ROC-AUC F0 (Table 16)                                      paper=      0.998144  now=      0.998144
   OK       UNSW-NB15    re

### If JupyterHub freezes or the browser is closed
Everything is saved while the run progresses, in `release/audit/results/revision_r1/`:
- `revision_r1_report_PARTIAL.txt` / `.json` — all sections finished so far (updated after every section);
- `sections/` — one checkpoint per finished section; `cache/` — predictions of every trained model;
- `run_log.txt` — time-stamped progress;
- at the end: `revision_r1_report.txt` and `revision_r1_results.json` (the PARTIAL files are then removed).

**To continue after an interruption:** run the first code cell again. Finished sections are loaded from their checkpoints and finished models from the cache; only the unfinished part is computed. The final report is identical to an uninterrupted run.

**To only look at the results:** run the cell below — it prints the saved report and computes nothing.

In [ ]:
# Show the saved results WITHOUT recomputing anything (use after a freeze, a closed browser or a kernel restart)
import os
OUT = os.path.join(os.path.expanduser("~/notebooks/2 article/release"), "audit", "results", "revision_r1")
final, partial = os.path.join(OUT, "revision_r1_report.txt"), os.path.join(OUT, "revision_r1_report_PARTIAL.txt")
if os.path.exists(final):
    print(open(final, encoding="utf-8").read())
elif os.path.exists(partial):
    print("THE RUN HAS NOT FINISHED. Results of the finished sections:\n")
    print(open(partial, encoding="utf-8").read())
    print("\nLast lines of the run log:")
    print("".join(open(os.path.join(OUT, "run_log.txt"), encoding="utf-8").readlines()[-5:]))
else:
    print("No saved results yet in", OUT)